# MLLM_DriveLM_Evaluation

In [ ]:
import os
import yaml
from src.dataloader.drivelm_builder import DriveLMPromptBuilder
from src.models.mllm_evaluator import MLLMEvaluator
from src.utils.metrics import DigitalTwinMetrics

# 1. Load Configurations
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

# 2. Initialize Data Loader and Evaluator
builder = DriveLMPromptBuilder(
    dataroot=config["paths"]["dataroot"],
    max_image_size=tuple(config["evaluation"]["camera_resolution"])
)

evaluator = MLLMEvaluator(
    model_type=config["model"]["type"],
    model_name=config["model"]["name"]
)

# 3. Process Sample DriveLM Frame
sample_annotation = os.path.join(config["paths"]["annotations_dir"], "sample_frame_001.json")

if os.path.exists(sample_annotation):
    # Prepare API Payload
    payload = builder.format_openai_payload(sample_annotation)
    
    # Run Inference
    print(f"Running inference with {config['model']['name']}...")
    response_text = evaluator.predict(payload)
    
    # Parse Structured Outputs
    structured_output = DigitalTwinMetrics.parse_json_response(response_text)
    print("\n--- Parsed MLLM Graph-of-Thought Response ---")
    print(yaml.dump(structured_output))
else:
    print(f"Sample annotation not found at {sample_annotation}. Please check path configuration.")

In [3]:
import sys
import os

# Adds the project root (one level up if running from notebooks/) to sys.path
sys.path.append(os.path.abspath(".."))  

from src.dataloader.drivelm_builder import DriveLMPromptBuilder

# Initialize builder (will use dummy paths if image files don't exist yet)
builder = DriveLMPromptBuilder(dataroot="../data/raw/nuScenes")

# Load and parse mock annotation
json_path = "../data/raw/DriveLM/sample_frame_001.json"
frame_data = builder.load_annotation(json_path)

# Build text prompt
prompt = builder.build_user_prompt(frame_data)
print("--- Generated Prompt ---")
print(prompt)

--- Generated Prompt ---
Review the multi-view images and answer the following driving graph queries:

=== PERCEPTION STAGE ===
Q1: What are the important dynamic objects in the front view near the crosswalk?
Q2: What is the state of the traffic signal at the upcoming intersection?

=== PREDICTION STAGE ===
Q1: What is the expected trajectory of pedestrian [p082] over the next 3 seconds?
Q2: What is the collision risk if the ego vehicle maintains its current speed of 11.2 m/s?

=== PLANNING STAGE ===
Q1: What safe deceleration profile and action should the ego vehicle take?

Provide your answers in a valid JSON object matching the keys: 'perception', 'prediction', 'planning'.
